In [ ]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install transformers accelerate python-Levenshtein tqdm pillow datasets

In [ ]:
import os
import json
import ast
from datasets import load_dataset


os.chdir('/kaggle/working')

print("Downloading MP-DocVQA dataset...")
dataset = load_dataset("lmms-lab/MP-DocVQA", split="val")

os.makedirs("multidoc/images", exist_ok=True)
formatted_data = []

print("Extracting pages and formatting...")

for i, row in enumerate(dataset.select(range(100))):
    
    answers = ast.literal_eval(row["answers"]) if isinstance(row["answers"], str) else row["answers"]
    page_ids = []
    
    for page_num in range(1, 21):
        image_key = f"image_{page_num}"
        
        if image_key in row and row[image_key] is not None:
            page_id = f"doc_{i}_page_{page_num}"
            page_ids.append(page_id)
            row[image_key].save(f"multidoc/images/{page_id}.jpg")
    
    formatted_data.append({
        "questionId": row.get("questionId", str(i)),
        "question": row["question"],
        "answers": answers,
        "page_ids": page_ids
    })

with open("multi_val.json", "w") as f:
    json.dump({"data": formatted_data}, f, indent=4)

print("Multi-page data is ready!")

In [ ]:
import os
import json
import ast
import re
import torch
import Levenshtein
from PIL import Image
from tqdm import tqdm
from collections import defaultdict
from datasets import load_dataset
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

# -----------------------
# 1. Configuration Settings
# -----------------------
IMAGE_DIR = "multidoc/images"
MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"

# Process in chunks to manage resources gracefully
CHUNK_START = 0
CHUNK_END = 100  # Adjust as needed based on your dataset chunking plan


os.chdir('/kaggle/working')

# -----------------------
# 2. Dataset Preparation & Layout Extraction
# -----------------------
if not os.path.exists("multi_val.json"):
    print("Downloading MP-DocVQA dataset from Hugging Face...")
    
    dataset = load_dataset("lmms-lab/MP-DocVQA", split="val")
    
    os.makedirs(IMAGE_DIR, exist_ok=True)
    formatted_data = []

    print("Extracting multi-page structures and saving images...")
    for i, row in enumerate(dataset):
        answers = ast.literal_eval(row["answers"]) if isinstance(row["answers"], str) else row["answers"]
        page_ids = []
        
        
        for page_num in range(1, 21):
            image_key = f"image_{page_num}"
            if image_key in row and row[image_key] is not None:
                page_id = f"doc_{i}_page_{page_num}"
                page_ids.append(page_id)
                row[image_key].save(os.path.join(IMAGE_DIR, f"{page_id}.jpg"))
        
        formatted_data.append({
            "questionId": row.get("questionId", str(i)),
            "question": row["question"],
            "answers": answers,
            "page_ids": page_ids
        })

    with open("multi_val.json", "w") as f:
        json.dump({"data": formatted_data}, f, indent=4)
    print("Multi-page dataset configuration complete!")
else:
    print("Dataset already parsed and structured. Proceeding to evaluation...")

# -----------------------
# 3. Evaluation Metrics
# -----------------------
def exact_match_strict(pred, answers):
    pred = pred.lower().strip()
    answers = [a.lower().strip() for a in answers]
    return int(any(pred == a for a in answers))

def exact_match_relaxed(pred, answers):
    pred = pred.lower().strip()
    answers = [a.lower().strip() for a in answers]
    return int(any(pred == a or a in pred for a in answers))

def anls(pred, answers, tau=0.5):
    pred = pred.lower().strip()
    scores = []
    for gt in answers:
        gt = gt.lower().strip()
        dist = Levenshtein.distance(pred, gt)
        score = 1 - dist / max(len(pred), len(gt))
        scores.append(score if score >= tau else 0)
    return max(scores) if scores else 0

# -----------------------
# 4. Load Optimized Deep Learning Models
# -----------------------
print(f"Loading {MODEL_ID} onto GPU...")

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

# -----------------------
# 5. Core Operational Functions
# -----------------------
def get_page_summary(page_id):
    """Generates an internal spatial context summary of a single page layout."""
    image_path = os.path.join(IMAGE_DIR, page_id + ".jpg")
    if not os.path.exists(image_path):
        return ""

    image = Image.open(image_path).convert("RGB")
    image.thumbnail((1024, 1024))

    prompt = "Give a short summary of this page in 5 words."
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt}
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt"
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=10, do_sample=False)

    output_ids = output_ids[:, inputs["input_ids"].shape[1]:]
    summary = processor.batch_decode(output_ids, skip_special_tokens=True)[0].strip().lower()
    return summary

def select_best_page(page_ids, question):
    """Compares the current query text against extracted page contexts to score relevance."""
    best_page = None
    best_score = -1

    for page_id in page_ids:
        summary = get_page_summary(page_id)
        if not summary:
            continue

        score = sum(1 for w in question.lower().split() if w in summary)
        if score > best_score:
            best_score = score
            best_page = page_id

    return best_page

# -----------------------
# 6. Main Inference and Benchmarking Loop
# -----------------------
with open("multi_val.json", "r") as f:
    dataset = json.load(f)["data"]

# Slice out current chunk workload
sliced_dataset = dataset[CHUNK_START:CHUNK_END]
print(f"\nBeginning multi-page processing for active subset slice indices [{CHUNK_START}:{CHUNK_END}]")

results = []
total_em_strict = 0
total_em_relaxed = 0
total_anls = 0
count = 0

for sample in tqdm(sliced_dataset):
    question = sample["question"]
    answers = sample["answers"]
    page_ids = sample["page_ids"]

    # Step 1: Content-Based Visual Routing/Page Selection
    best_page = select_best_page(page_ids, question)
    if best_page is None:
        continue

    image_path = os.path.join(IMAGE_DIR, best_page + ".jpg")

    # Step 2: Optimized Extractive Answer Generation
    answer_image = Image.open(image_path).convert("RGB")
    answer_image.thumbnail((1024, 1024))

    prompt = (
        "Read the document carefully and answer exactly using text from the document. "
        "Give a short answer only. Do not explain.\n"
        f"Question: {question}"
    )

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": answer_image},
                {"type": "text", "text": prompt}
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt"
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=20, do_sample=False)

    output_ids = output_ids[:, inputs["input_ids"].shape[1]:]
    prediction = processor.batch_decode(output_ids, skip_special_tokens=True)[0].strip()

    # Score evaluations
    em_strict = exact_match_strict(prediction, answers)
    em_relaxed = exact_match_relaxed(prediction, answers)
    anls_score = anls(prediction, answers)

    total_em_strict += em_strict
    total_em_relaxed += em_relaxed
    total_anls += anls_score
    count += 1

    results.append({
        "questionId": sample["questionId"],
        "question": question,
        "prediction": prediction,
        "answers": answers,
        "page_selected": best_page,
        "page_index_selected": page_ids.index(best_page) if best_page in page_ids else -1,
        "exact_match_strict": em_strict,
        "exact_match_relaxed": em_relaxed,
        "anls": anls_score
    })

# -----------------------
# 7. Metrics Aggregation & Saving Reports
# -----------------------
accuracy_strict = total_em_strict / count if count else 0
accuracy_relaxed = total_em_relaxed / count if count else 0
mean_anls = total_anls / count if count else 0

print("\n===== FINAL CHUNK RUN RESULTS =====")
print(f"Samples Evaluated: {count}")
print(f"Strict EM Accuracy:  {accuracy_strict:.4f}")
print(f"Relaxed EM Accuracy: {accuracy_relaxed:.4f}")
print(f"ANLS Score Overall:  {mean_anls:.4f}")

output_report_file = f"mp_docvqa_smart_selection_{CHUNK_START}_{CHUNK_END}.json"
with open(output_report_file, "w") as f:
    json.dump({
        "overall": {
            "exact_match_strict": accuracy_strict,
            "exact_match_relaxed": accuracy_relaxed,
            "anls": mean_anls
        },
        "results": results
    }, f, indent=2)

print(f"Successfully compiled and saved analysis metadata target to: {output_report_file}")